## Model Tuning

In [1]:
import pandas as pd
from sklearn.model_selection import (
    train_test_split, GridSearchCV, KFold, StratifiedKFold
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    make_scorer, recall_score, precision_score, f1_score,
    accuracy_score, classification_report, confusion_matrix
)

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df=pd.read_csv("../data/student_dataset/student-mat.csv", sep=";")
X=df.drop(columns=["G3"])
y_grade= df["G3"]
y_pass=(df["G3"]>=10).astype(int)

In [2]:
#Train/Test Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg=train_test_split(
    X, y_grade, test_size=0.2, random_state=42
)

X_train_classi, X_test_classi, y_train_classi, y_test_classi=train_test_split(
    X, y_pass, test_size=0.2, random_state=42, stratify=y_pass
)

In [3]:
#Preprocessing
numerical_cols=X.select_dtypes(include="number").columns
categorical_cols=X.select_dtypes(include=["str", "object", "category"]).columns

preprocessor=ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [4]:
#Tune Regression Models
regression_models={
    "dummy": (Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", DummyRegressor(strategy="mean"))
    ]),{}
    ),
    "linear":(Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]),
    {})
    ,
    "ridge":(Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", Ridge())
    ]),
    {"regressor__alpha": [0.01, 0.1, 1, 10, 100]}
    )
    ,
    "lasso": (Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", Lasso(max_iter=10000))
    ]), 
    {"regressor__alpha": [0.001, 0.01, 0.1, 1]}
    ),
    "elasticnet": (Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", ElasticNet(max_iter=10000))
    ]),  {
            "regressor__alpha": [0.001, 0.01, 0.1, 1],
            "regressor__l1_ratio": [0.2, 0.5, 0.8]
        }
        )
}
reg_cv=KFold(n_splits=5, shuffle=True, random_state=42)
reg_searches={}
for name, (pipeline, grid) in regression_models.items():
    search=GridSearchCV(
        pipeline, grid, cv=reg_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=1
    )
    search.fit(X_train_reg, y_train_reg)
    reg_searches[name]=search
    print(name, "CV MAE: ", -search.best_score_,
          "Best Settings: ", search.best_params_)

dummy CV MAE:  3.4028651224982744 Best Settings:  {}
linear CV MAE:  1.392734456225702 Best Settings:  {}
ridge CV MAE:  1.3283886791827584 Best Settings:  {'regressor__alpha': 10}
lasso CV MAE:  1.1525425328302317 Best Settings:  {'regressor__alpha': 0.1}
elasticnet CV MAE:  1.1678929360834704 Best Settings:  {'regressor__alpha': 0.1, 'regressor__l1_ratio': 0.8}


In [5]:
# Classification cross-validation and scoring
classi_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
class_scoring = {
    "recall_not_pass": make_scorer(recall_score, pos_label=0),
    "precision_not_pass": make_scorer(precision_score, pos_label=0, zero_division=0),
    "f1_not_pass": make_scorer(f1_score, pos_label=0, zero_division=0),
    "accuracy": "accuracy"
}


## Tune classification models

Compare the four classifiers using five-fold stratified cross-validation. The search selects settings by recall for class 0 (did not pass), while also recording precision, F1, and accuracy.

In [6]:
classification_models = {
    "logistic": (LogisticRegression(max_iter=10000), {
        "classifier__C": [0.1, 1, 10]
    }),
    "knn": (KNeighborsClassifier(), {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"]
    }),
    "decision_tree": (DecisionTreeClassifier(random_state=42), {
        "classifier__max_depth": [3, 5, 7, None],
        "classifier__min_samples_leaf": [1, 2, 4]
    }),
    "random_forest": (RandomForestClassifier(random_state=42), {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4]
    })
}

classi_searches = {}
classi_rows = []
for name, (estimator, grid) in classification_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    search = GridSearchCV(
        pipeline,
        grid,
        cv=classi_cv,
        scoring=class_scoring,
        refit="recall_not_pass",
        n_jobs=1
    )
    search.fit(X_train_classi, y_train_classi)
    classi_searches[name] = search
    i = search.best_index_
    cv_results = search.cv_results_
    classi_rows.append({
        "model": name,
        "cv_recall_not_pass": cv_results["mean_test_recall_not_pass"][i],
        "cv_precision_not_pass": cv_results["mean_test_precision_not_pass"][i],
        "cv_f1_not_pass": cv_results["mean_test_f1_not_pass"][i],
        "cv_accuracy": cv_results["mean_test_accuracy"][i],
        "best_params": search.best_params_
    })

classi_results_df = pd.DataFrame(classi_rows).sort_values(
    "cv_recall_not_pass", ascending=False
)
classi_results_df

,model,cv_recall_not_pass,cv_precision_not_pass,cv_f1_not_pass,cv_accuracy,best_params
2,decision_tree,0.913333,0.880369,0.896288,0.930258,"{'classifier__max_depth': 5, 'classifier__min_..."
0,logistic,0.883810,0.879676,0.880275,0.920833,{'classifier__C': 10}
3,random_forest,0.883810,0.888318,0.884081,0.924058,"{'classifier__max_depth': None, 'classifier__m..."
1,knn,0.520000,0.835967,0.639513,0.809921,"{'classifier__n_neighbors': 7, 'classifier__we..."


## Final test-set evaluation

The searches above use only training data. Evaluate the selected models on the held-out test sets once, after choosing by cross-validation.

In [7]:
# Best regression model: highest negative-MAE score equals lowest MAE
best_reg_name = max(reg_searches, key=lambda name: reg_searches[name].best_score_)
best_reg = reg_searches[best_reg_name].best_estimator_
reg_predictions = best_reg.predict(X_test_reg)
print("Best regression model:", best_reg_name)
print("Test MAE:", mean_absolute_error(y_test_reg, reg_predictions))
print("Test RMSE:", mean_squared_error(y_test_reg, reg_predictions) ** 0.5)
print("Test R2:", r2_score(y_test_reg, reg_predictions))

# Best classifier: selected by cross-validation recall for class 0
best_classi_name = max(
    classi_searches,
    key=lambda name: classi_searches[name].best_score_
)
best_classi = classi_searches[best_classi_name].best_estimator_
classi_predictions = best_classi.predict(X_test_classi)
print("\nBest classifier:", best_classi_name)
print("Test confusion matrix (rows/columns are 0=did not pass, 1=passed):")
print(confusion_matrix(y_test_classi, classi_predictions, labels=[0, 1]))
print(classification_report(
    y_test_classi,
    classi_predictions,
    labels=[0, 1],
    target_names=["did not pass", "passed"],
    zero_division=0
))

Best regression model: lasso
Test MAE: 1.2817964028756461
Test RMSE: 2.0928000928552164
Test R2: 0.7864032122727267

Best classifier: decision_tree
Test confusion matrix (rows/columns are 0=did not pass, 1=passed):
[[24  2]
 [ 8 45]]
              precision    recall  f1-score   support

did not pass       0.75      0.92      0.83        26
      passed       0.96      0.85      0.90        53

    accuracy                           0.87        79
   macro avg       0.85      0.89      0.86        79
weighted avg       0.89      0.87      0.88        79



## Findings

After running the notebook, summarize the selected regression model and its test MAE/RMSE/R². For classification, explain the test recall for “did not pass” and the number of missed at-risk students from the confusion matrix. Mention the tradeoff with precision (students who passed but were flagged). These results use one held-out split and should be treated as a learning exercise, not as a basis for decisions about real students.